# PAPERS — SUBFIELD DATA (OPENALEX)

### Input data structure (required)
Set `data_dir_papers` in the first code cell. Inside that folder, provide subfield parquet files.

File naming
- `merged_df_subfield_{ID}.parquet` (one file per subfield ID)

Required columns used in this notebook
- `date`: publication date (string or datetime)
- `topics`: array‑like; used only to filter out empty records
- `keywords`: array‑like; used to build topic combinations (order‑independent)
- `authors`: array‑like; we use only the first author as a proxy for creator counts

### What the filtered dataset represents
After loading, we keep only rows where `topics` and `authors` are non‑empty. This ensures
that every record contributes to both topic‑combination statistics and author counts.

### What this notebook produces
- `delta_rows_papers.csv` saved in `output_dir`
- `openalex_combo_freq.csv` saved in `output_dir`
- `openalex_author_freq.csv` saved in `output_dir`
- `reduced_openalex_log.csv` and `reduced_openalex_lin.csv` saved in `output_dir`


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Physics and Astronomy, and Mathematics subfields
subfields = [14, 25, 37, 39, 54, 114, 131, 246, 110, 142, 164, 172, 191, 193, 206, 210, 216, 248]
start_date = 1980
end_date = 2020

data_dir_papers = Path("../../openalex/data/subfield")
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Load and concatenate all subfields
# Each parquet corresponds to a single OpenAlex subfield ID
# Each parquet corresponds to one OpenAlex subfield ID

dfs = []
for subfield_index in subfields:
    path = data_dir_papers / f"merged_df_subfield_{subfield_index}.parquet"
    df = pd.read_parquet(path).dropna()
    df['subfield'] = subfield_index
    dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)

# Keep only rows with non-empty topics and authors
# These are the minimum fields required for our statistics
# We only keep records that can contribute to topic/author statistics
merged_df = merged_df[
    merged_df['topics'].apply(lambda x: isinstance(x, np.ndarray) and len(x) > 0) &
    merged_df['authors'].apply(lambda x: isinstance(x, np.ndarray) and len(x) > 0)
].reset_index(drop=True)

# Extract year
# Used for filtering and yearly aggregation
# This is used to filter the time window and build yearly deltas
merged_df['year'] = pd.to_datetime(merged_df['date'], errors='coerce').dt.year

# Filter by year range
merged_df = merged_df[(merged_df["year"] >= start_date) & (merged_df["year"] <= end_date)]

# Count publications per year (used for delta_rows)
year_counts = merged_df.groupby('year').size()


/tmp/ipykernel_21489/917352407.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  merged_df['year'] = pd.to_datetime(merged_df['date'], errors='coerce').dt.year


## Preprocess: exclude first authors with too many papers per year

After loading and filtering all subfields into `merged_df`, we identify **first authors** who publish more than `max_papers_per_year` papers in any single year. Those authors are excluded from the analysis: all rows where they are the first author are removed from `merged_df` before building cumulative series and frequencies.


In [2]:
import numpy as np

max_papers_per_year = 100

# Identify first authors
merged_df['first_author'] = merged_df['authors'].apply(lambda a: a[0] if isinstance(a, (list, tuple, np.ndarray)) and len(a) > 0 else None)

# Count papers per (year, first_author)
counts = (
    merged_df
    .dropna(subset=['first_author', 'year'])
    .groupby(['year', 'first_author'])
    .size()
)

excluded_authors = set(counts[counts > max_papers_per_year].reset_index()['first_author'])

found_authors = merged_df['first_author'].dropna().nunique()
removed_authors = len(excluded_authors)
remaining_authors = found_authors - removed_authors

print(f"Total first authors found: {found_authors:,}")
print(f"Total first authors removed: {removed_authors:,}")
print(f"Total first authors after removal: {remaining_authors:,}")

# Filter merged_df: remove rows where first author is excluded
before_rows = len(merged_df)
merged_df = merged_df[~merged_df['first_author'].isin(excluded_authors)].reset_index(drop=True)
after_rows = len(merged_df)

print(f"Rows before exclusion: {before_rows:,}")
print(f"Rows after exclusion: {after_rows:,}")

# Recompute yearly counts after exclusion
year_counts = merged_df.groupby('year').size()


Total first authors found: 1,673,411
Total first authors removed: 58
Total first authors after removal: 1,673,353
Rows before exclusion: 7,047,964
Rows after exclusion: 7,007,499


## BUILD CUMULATIVE SERIES AND FREQUENCIES

This cell goes through the filtered papers in chronological order and builds:
- cumulative number of unique topic combinations over time
- cumulative number of unique first authors over time
- frequency tables for topic combinations and first authors

It also computes yearly deltas:
- `delta_rows` is the number of papers per year
- `num_first_authors` is the cumulative number of unique first authors at year end


In [3]:
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path


# Extract year safely and sort
# Sorting is important to build cumulative series in chronological order

def extract_year(date):
    try:
        return int(str(date)[:4])
    except Exception:
        return np.nan

merged_df["year"] = merged_df["date"].apply(extract_year)
merged_df = merged_df.dropna(subset=["year"]).sort_values("year").reset_index(drop=True)

seen_combinations = set()
combo_freq = {}
num_unique_combinations = []

seen_authors = set()
author_freq = {}
num_unique_authors = []

row_indices = []
years = []

for i, row in tqdm(merged_df.iterrows(), total=len(merged_df), desc="Processing topic combinations"):
    keywords = row['keywords']
    if isinstance(keywords, (list, tuple, np.ndarray)) and len(keywords) > 0:
        # Use frozenset so that order does not matter in the combination
        combo = frozenset(keywords)
        combo_freq[combo] = combo_freq.get(combo, 0) + 1
        if combo not in seen_combinations:
            seen_combinations.add(combo)
    num_unique_combinations.append(len(seen_combinations))

    # First author only
    # Use only the first author as a proxy for creator counts
    authors = row['authors']
    if isinstance(authors, (list, tuple, np.ndarray)) and len(authors) > 0:
        first_author = authors[0]
        author_freq[first_author] = author_freq.get(first_author, 0) + 1
        if first_author not in seen_authors:
            seen_authors.add(first_author)
    num_unique_authors.append(len(seen_authors))

    row_indices.append(i + 1)
    years.append(row["year"])

years = np.array(years)
num_unique_authors = np.array(num_unique_authors)

# Yearly delta rows (publication counts)
# Uses the filtered dataset so the deltas are consistent with the series above
# We take the publication counts from the filtered dataset
unique_years = year_counts.index.values

authors_last_per_year = []
for y in unique_years:
    mask = years == y
    if np.any(mask):
        authors_last_per_year.append(num_unique_authors[mask][-1])
    else:
        authors_last_per_year.append(np.nan)

x = np.array(authors_last_per_year)
y = year_counts.values

delta_rows_df = pd.DataFrame({'delta_rows': y, 'num_first_authors': x})
delta_rows_df.to_csv(output_dir / "delta_rows_papers.csv", index=False)

# Save combination frequency table
# Rank by frequency (descending)
combo_freq_sorted = sorted(combo_freq.items(), key=lambda x: x[1], reverse=True)
df_combo_freq = pd.DataFrame({
    'rank': np.arange(1, len(combo_freq_sorted) + 1),
    'combo': [list(k) for k, v in combo_freq_sorted],
    'frequency': [v for k, v in combo_freq_sorted]
})
df_combo_freq.to_csv(output_dir / "openalex_combo_freq.csv", index=False)

# Save author frequency table
# Rank by frequency (descending)
author_freq_sorted = sorted(author_freq.items(), key=lambda x: x[1], reverse=True)
df_author_freq = pd.DataFrame({
    'rank': np.arange(1, len(author_freq_sorted) + 1),
    'author': [k for k, v in author_freq_sorted],
    'frequency': [v for k, v in author_freq_sorted]
})
df_author_freq.to_csv(output_dir / "openalex_author_freq.csv", index=False)


Processing topic combinations: 100%|██████████| 7007499/7007499 [09:07<00:00, 12794.19it/s]


## REDUCED SERIES (LOG + LINEAR)

This cell reduces the full cumulative series into two 10k‑point samples:
- log-spaced time indices (for early‑time detail)
- linearly spaced time indices (uniform over the series)

The reduced files are used in later modules for plotting and model comparison.


In [4]:
import numpy as np
import pandas as pd

n_points = 10**4  # number of points kept in the reduced series
N = len(years)

# Log-spaced indices give higher resolution at early times
log_indices = np.unique(np.logspace(0, np.log10(N), n_points, dtype=int) - 1)
# Linear indices sample uniformly over the full series
lin_indices = np.linspace(0, N - 1, n_points, dtype=int)

years_log = np.array(years)[log_indices]
num_unique_combinations_log = np.array(num_unique_combinations)[log_indices]
num_unique_authors_log = np.array(num_unique_authors)[log_indices]
row_indices_log = np.array(row_indices)[log_indices]

years_lin = np.array(years)[lin_indices]
num_unique_combinations_lin = np.array(num_unique_combinations)[lin_indices]
num_unique_authors_lin = np.array(num_unique_authors)[lin_indices]
row_indices_lin = np.array(row_indices)[lin_indices]

df_log = pd.DataFrame({
    'years': years_log,
    'num_unique_combinations': num_unique_combinations_log,
    'num_unique_authors': num_unique_authors_log,
    'row_indices': row_indices_log
})
df_log.to_csv(output_dir / "reduced_openalex_log.csv", index=False)

df_lin = pd.DataFrame({
    'years': years_lin,
    'num_unique_combinations': num_unique_combinations_lin,
    'num_unique_authors': num_unique_authors_lin,
    'row_indices': row_indices_lin
})
df_lin.to_csv(output_dir / "reduced_openalex_lin.csv", index=False)


## AUTHOR DATES (PAPERS)

This cell builds a per-author event list from the already loaded `merged_df`.
We use the first author and the topic list (OpenAlex `topics`) to assign novelty types.
The result is saved as a pickle in `output_dir/`.


In [ ]:
from collections import defaultdict
from tqdm import tqdm
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

first_year = 1980
last_year = 2020

# Filter the already-loaded merged_df

df_filtered = merged_df[
    (merged_df["year"] >= first_year) &
    (merged_df["year"] <= last_year)
].copy()

print(f"Total publications considered ({first_year}-{last_year}): {len(df_filtered):,}")

# --- Step 1: collect all publications for global novelty ---
all_publications = []

for row in tqdm(df_filtered.itertuples(index=False),
                total=len(df_filtered),
                desc="Collecting OpenAlex publications"):

    date = pd.to_datetime(row.date, errors="coerce")
    authors = row.authors
    topics = row.topics

    if pd.isna(date):
        continue
    if not isinstance(authors, (list, np.ndarray)) or len(authors) == 0:
        continue
    if not isinstance(topics, (list, np.ndarray)) or len(topics) == 0:
        continue

    first_author = str(authors[0]).strip()
    if not first_author:
        continue

    topic_tuple = tuple(sorted(map(str, topics)))

    all_publications.append({
        "date": date,
        "author": first_author,
        "topics": topic_tuple
    })

print(f"Total valid publications: {len(all_publications):,}")

# --- Step 2: sort by date ---
all_publications = sorted(all_publications, key=lambda x: x["date"])

# --- Step 3: assign novelty types (global vs individual) ---

global_seen = set()
individual_seen = defaultdict(set)

author_dates = defaultdict(list)

for entry in all_publications:
    author = entry["author"]
    date = entry["date"]
    topic_combo = entry["topics"]

    if topic_combo not in global_seen:
        novelty = "global_novelty"
        global_seen.add(topic_combo)
        individual_seen[author].add(topic_combo)
    else:
        if topic_combo not in individual_seen[author]:
            novelty = "individual_novelty"
            individual_seen[author].add(topic_combo)
        else:
            novelty = "no_novelty"

    author_dates[author].append({
        "date": date,
        "topics": list(topic_combo),
        "novelty_type": novelty
    })

# --- Step 4: enrich with time features and cumulative novelty counters ---
max_years = last_year - first_year

for author, events in author_dates.items():
    if not events:
        continue

    events = sorted(events, key=lambda x: x["date"])
    t0 = pd.to_datetime(events[0]["date"])

    cumulative_individual = 0
    cumulative_global = 0

    for i, ev in enumerate(events):
        t = pd.to_datetime(ev["date"])

        if ev["novelty_type"] == "global_novelty":
            cumulative_global += 1
            cumulative_individual += 1
        elif ev["novelty_type"] == "individual_novelty":
            cumulative_individual += 1

        ev["cumulative_individual_novelty"] = cumulative_individual
        ev["cumulative_global_novelty"] = cumulative_global

        ev["year"] = t.year
        ev["year_relative"] = t.year - t0.year

        rel_years = (t - t0).total_seconds() / (365.25 * 24 * 3600)
        ev["relative_year"] = rel_years
        ev["cumulative_count"] = i + 1

        day_of_year = t.timetuple().tm_yday
        abs_year_decimal = t.year + (day_of_year - 1) / 365.25
        ev["abs_year_decimal"] = abs_year_decimal

        scaled = (abs_year_decimal - first_year) * (max_years / (last_year - first_year))
        ev["scaled_year_global"] = scaled

    author_dates[author] = events

# --- Save ---
output_path = output_dir / f"openalex_author_dates_{first_year}_{last_year}.pkl"
with open(output_path, "wb") as f:
    pickle.dump(author_dates, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved author dates to {output_path}")


Total publications considered (1980-2020): 7,007,499


Total valid publications: 7,007,499


## AUTHOR INTERTIMES (PAPERS)

This cell loads the author-dates dictionary and computes inter-event times
for four cases: all events, global novelty only, any novelty, and no novelty.
The output is saved as a pickle in `output_dir/`.


In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path

input_path = output_dir / "openalex_author_dates_1980_2020.pkl"

with open(input_path, "rb") as f:
    author_dates = pickle.load(f)


def compute_intervals_with_dates(events_list, author_name):
    '''Compute inter-event times with start/end dates and author name.'''
    if len(events_list) < 2:
        return []

    dates = sorted([pd.to_datetime(ev["date"]) for ev in events_list])

    intervals = []
    for d1, d2 in zip(dates[:-1], dates[1:]):
        dt = (d2 - d1).days / 365.25
        if 0 < dt < 50:
            intervals.append({
                "interval_years": dt,
                "start_date": d1,
                "end_date": d2,
                "author": author_name
            })

    return intervals

all_intervals = []
global_only_intervals = []
novelties_intervals = []
no_novelty_intervals = []

for author, events in author_dates.items():
    all_intervals.extend(compute_intervals_with_dates(events, author))

    events_global = [ev for ev in events if ev["novelty_type"] == "global_novelty"]
    global_only_intervals.extend(compute_intervals_with_dates(events_global, author))

    events_novel = [ev for ev in events if ev["novelty_type"] in ("global_novelty", "individual_novelty")]
    novelties_intervals.extend(compute_intervals_with_dates(events_novel, author))

    events_none = [ev for ev in events if ev["novelty_type"] == "no_novelty"]
    no_novelty_intervals.extend(compute_intervals_with_dates(events_none, author))

intervals_dict = {
    "all_intervals": all_intervals,
    "global_only_intervals": global_only_intervals,
    "novelties_intervals": novelties_intervals,
    "no_novelty_intervals": no_novelty_intervals,
}

output_path = output_dir / "openalex_intervals.pkl"
with open(output_path, "wb") as f:
    pickle.dump(intervals_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved intertimes to {output_path}")
